In [14]:
import numpy as np
from scipy.optimize import minimize

In [15]:
dt, N = 0.02, 3                      # timestep, number of steps

P0 = np.array([0.9, 3.0])            # previous waypoint
P1 = np.array([1.0, 4.1])            # next waypoint
d  = P1 - P0                         # path direction vector
L  = np.linalg.norm(d)               # segment length
psi = np.arctan2(d[1], d[0])         # desired heading (rad)

x0  = np.array([0.1, 3.1])           # car position
th0 = np.pi/2                        # car heading (assumed pointing up)
v0, vd = 5.5, 10.0                   # current and desired speed

print("Desired heading (deg):", np.degrees(psi))

Desired heading (deg): 84.8055710922652


In [16]:
wrap = lambda a: (a + np.pi) % (2*np.pi) - np.pi

In [17]:
def J(u, w=(1, 1, 1)):
    a, om = u
    x = x0.copy(); th = th0; v = v0; tot = 0
    for _ in range(N):
        # move the car one step
        x = x + v*dt*np.array([np.cos(th), np.sin(th)])
        th += om*dt
        v  += a*dt
        # errors
        eh  = wrap(th - psi)
        ect = (d[0]*(x[1]-P0[1]) - d[1]*(x[0]-P0[0])) / L
        ev  = v - vd
        # add weighted squared errors
        tot += w[0]*eh**2 + w[1]*ect**2 + w[2]*ev**2
    return tot

In [18]:
u0 = [1.3, 0.23]
print("J before:", J(u0))

J before: 61.434591178736895


In [19]:
res_unb = minimize(J, u0, method='BFGS')
print("a* =", res_unb.x[0], " omega* =", res_unb.x[1])
print("J after (unbounded):", res_unb.fun)

a* = 96.43448556472813  omega* = -3.8787158953993432
J after (unbounded): 10.686088900729267


In [20]:
bounds = [(-8, 4), (-0.8, 0.8)]      # (a limits), (omega limits)
res_b = minimize(J, u0, method='L-BFGS-B', bounds=bounds)
print("a* =", res_b.x[0], " omega* =", res_b.x[1])
print("J after (bounded):", res_b.fun)

a* = 4.0  omega* = -0.8
J after (bounded): 58.564948609327075


In [21]:
weights = [(1, 1, 1), (1, 10, 1), (1, 1, 0.01)]
for w in weights:
    r = minimize(lambda u: J(u, w), u0, method='L-BFGS-B', bounds=bounds)
    print(w, "-> a =", round(r.x[0], 3), " omega =", round(r.x[1], 3), " J =", round(r.fun, 3))

(1, 1, 1) -> a = 4.0  omega = -0.8  J = 58.565
(1, 10, 1) -> a = 4.0  omega = -0.8  J = 76.876
(1, 1, 0.01) -> a = 4.0  omega = -0.8  J = 2.611


In [22]:
%pip install scipy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip
